# FIRE Zero-Shot Test
**FIRE** (CVPR 2025) — Frequency-Guided Reconstruction Error for diffusion fake detection.


## Cell 1 — Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
ZIP_PATH = '/content/drive/MyDrive/Adv_CV_Project/openfake_subset.zip'  # <-- CHANGE THIS if needed
assert os.path.exists(ZIP_PATH), f'Zip not found at {ZIP_PATH}. Check the path.'
print(f'Found zip: {ZIP_PATH}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found zip: /content/drive/MyDrive/Adv_CV_Project/openfake_subset.zip


## Cell 2 — Extract Dataset

In [2]:
import zipfile, os

EXTRACT_DIR = '/content/dataset'
os.makedirs(EXTRACT_DIR, exist_ok=True)

print('Extracting zip (this may take a few minutes)...')
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(EXTRACT_DIR)
print('Done.')

for root, dirs, files in os.walk(EXTRACT_DIR):
    depth = root.replace(EXTRACT_DIR, '').count(os.sep)
    if depth > 2:
        continue
    indent = '  ' * depth
    print(f'{indent}{os.path.basename(root)}/')
    if depth == 2:
        print(f'{indent}  ... ({len(files)} files)')


Extracting zip (this may take a few minutes)...
Done.
dataset/
  images/


## Cell 3 — Load Metadata and Fix Paths

In [3]:
import pandas as pd

METADATA_PATH = '/content/dataset/metadata.csv'
df = pd.read_csv(METADATA_PATH)

df['image_path'] = df['image_path'].str.replace(
    '/content/openfake_subset/images',
    '/content/dataset/images',
    regex=False
)

sample = df['image_path'].iloc[0]
print(f'Remapped path sample : {sample}')
print(f'File exists          : {os.path.exists(sample)}')
print(f'\nShape: {df.shape}')
print(f'\nLabels:\n{df["label"].value_counts()}')

Remapped path sample : /content/dataset/images/0.jpg
File exists          : True

Shape: (8989, 4)

Labels:
label
real                3000
stable_diffusion    3000
midjourney          2989
Name: count, dtype: int64


## Cell 4 — Split into Train / Val / Test (80 / 10 / 10)


In [4]:
import pandas as pd, os, shutil

METADATA_PATH = '/content/dataset/metadata.csv'
OUT_BASE = '/content/splits'

df = pd.read_csv(METADATA_PATH)
df['image_path'] = df['image_path'].str.replace(
    '/content/openfake_subset/images',
    '/content/dataset/images', regex=False
)

df['split_label'] = df['label'].apply(lambda x: 'real' if str(x).lower() == 'real' else 'fake')

real_df = df[df['split_label'] == 'real'].copy()
fake_df = df[df['split_label'] == 'fake'].copy()

def split_df(dataframe, splits=(0.8, 0.1, 0.1)):
    dataframe = dataframe.sample(frac=1, random_state=42).reset_index(drop=True)
    n = len(dataframe)
    n_train = int(n * splits[0])
    n_val   = int(n * splits[1])
    return {
        'train': dataframe.iloc[:n_train],
        'val'  : dataframe.iloc[n_train:n_train+n_val],
        'test' : dataframe.iloc[n_train+n_val:]
    }

def copy_split(split_dict, label, out_base):
    for split_name, sdf in split_dict.items():
        dst = os.path.join(out_base, split_name, label)
        os.makedirs(dst, exist_ok=True)
        copied, missing = 0, 0
        for _, row in sdf.iterrows():
            src = row['image_path']
            if not os.path.exists(src):
                missing += 1
            else:
                shutil.copy(src, os.path.join(dst, os.path.basename(src)))
                copied += 1
        print(f'  {split_name}/{label}: {copied} copied, {missing} missing')

print('Copying real images...')
copy_split(split_df(real_df), 'real', OUT_BASE)
print('Copying fake images...')
copy_split(split_df(fake_df), 'fake', OUT_BASE)

print('\nFinal split summary:')
for split_name in ['train', 'val', 'test']:
    r = len(os.listdir(f'{OUT_BASE}/{split_name}/real'))
    f = len(os.listdir(f'{OUT_BASE}/{split_name}/fake'))
    print(f'  {split_name:5s} → real: {r:4d}  fake: {f:4d}  total: {r+f}')

Copying real images...
  train/real: 2400 copied, 0 missing
  val/real: 300 copied, 0 missing
  test/real: 300 copied, 0 missing
Copying fake images...
  train/fake: 4791 copied, 0 missing
  val/fake: 598 copied, 0 missing
  test/fake: 600 copied, 0 missing

Final split summary:
  train → real: 2400  fake: 4791  total: 7191
  val   → real:  300  fake:  598  total: 898
  test  → real:  300  fake:  600  total: 900


## Cell 5 — Clone FIRE and Install Dependencies


In [5]:
%%bash
cd /content
if [ -d "FIRE" ]; then
  rm -rf FIRE
fi
git clone https://github.com/Chuchad/FIRE.git

# ── Step 1: Install compatible versions
pip install -q "diffusers==0.27.2" \
               "transformers==4.44.0" \
               "accelerate>=0.20.0" \
               typed-argument-parser torchmetrics

# ── Step 2: Patch diffusers dynamic_modules_utils.py
DIFFUSERS_FILE=$(find /usr/local/lib/python3.12/dist-packages/diffusers/utils -name "dynamic_modules_utils.py" 2>/dev/null | head -1)

if [ -z "$DIFFUSERS_FILE" ]; then
  echo "ERROR: Could not find dynamic_modules_utils.py"
else
  echo "Found: $DIFFUSERS_FILE"
  sed -i 's/from huggingface_hub import cached_download, hf_hub_download, model_info/from huggingface_hub import hf_hub_download, model_info/' "$DIFFUSERS_FILE"
  sed -i 's/cached_download(/hf_hub_download(/g' "$DIFFUSERS_FILE"
  echo "Patched dynamic_modules_utils.py"
  grep -n "cached_download" "$DIFFUSERS_FILE" && echo "WARNING: cached_download still present" || echo "Verified: cached_download removed"
fi

# ── Step 3: Patch FIRE network_utils.py
python - << 'PATCHEOF'
import re

path = '/content/FIRE/utils/network_utils.py'
with open(path, 'r') as f:
    content = f.read()

new_impl = """# retrieve_latents patched inline for diffusers compatibility
def retrieve_latents(encoder_output, generator=None):
    if hasattr(encoder_output, 'latent_dist'):
        return encoder_output.latent_dist.sample(generator=generator) if generator else encoder_output.latent_dist.sample()
    elif hasattr(encoder_output, 'latents'):
        return encoder_output.latents
    else:
        raise AttributeError('encoder_output has neither latent_dist nor latents')"""

pattern = r'from diffusers\.pipelines\.stable_diffusion\.pipeline_stable_diffusion_img2img import\s*\(\s*retrieve_latents,?\s*\)'

if re.search(pattern, content):
    content = re.sub(pattern, new_impl, content)
    with open(path, 'w') as f:
        f.write(content)
    print('Patched FIRE network_utils.py successfully')
else:
    print('ERROR: Pattern not found in network_utils.py')
PATCHEOF

# ── Step 4: Patch FIRE to skip SD pipeline download at import time ────────────
# Without this patch, every FIRE_model() call downloads the full SD pipeline
# (~4 GB) even though it is not needed for inference or fine-tuning.
python - << 'PATCHEOF2'
import re

path = '/content/FIRE/utils/network_utils.py'
with open(path, 'r') as f:
    content = f.read()

old_sd_load = 'self.sd_pipe = StableDiffusionPipeline.from_pretrained('
new_sd_load = 'self.sd_pipe = None  # SD pipeline loaded lazily — not needed for inference\n        if False: self.sd_pipe = StableDiffusionPipeline.from_pretrained('

if old_sd_load in content:
    content = content.replace(old_sd_load, new_sd_load)
    with open(path, 'w') as f:
        f.write(content)
    print('Patched FIRE_model SD pipeline loading (lazy init)')
else:
    print('SD pipeline patch not needed or pattern changed — skipping')
    for i, line in enumerate(content.split('\n'), 1):
        if 'StableDiffusion' in line or 'sd_pipe' in line:
            print(f'  {i}: {line}')
PATCHEOF2

echo 'All patches done. Please restart runtime now before continuing.'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 126.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 128.2 MB/s eta 0:00:00
Found: /usr/local/lib/python3.12/dist-packages/diffusers/utils/dynamic_modules_utils.py
Patched dynamic_modules_utils.py
Verified: cached_download removed
Patched FIRE network_utils.py successfully
SD pipeline patch not needed or pattern changed — skipping
  126:     pipe = diffusers.StableDiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5",
All patches done. Please restart runtime now before continuing.


Cloning into 'FIRE'...


## Cell 6 — Download Pretrained Checkpoint
**lsun_bedroom+adm**: https://drive.google.com/file/d/1cbaoRMJH9y-reOesTygSvuAU7uz6dNGm/view



In [5]:
import os

CKPT_DRIVE_PATH = '/content/drive/MyDrive/Adv_CV_Project/lsunbedroom_w_adm.pt'

CKPT_LOCAL = '/content/FIRE/ckpt/lsunbedroom_w_adm.pt'
os.makedirs('/content/FIRE/ckpt', exist_ok=True)

if os.path.exists(CKPT_DRIVE_PATH):
    import shutil
    shutil.copy(CKPT_DRIVE_PATH, CKPT_LOCAL)
    print(f'Checkpoint copied from Drive: {CKPT_LOCAL}')
else:
    print(f'File not found at {CKPT_DRIVE_PATH}')
    print('Please download the checkpoint from the FIRE repo and re-run this cell.')
    print('Download: https://drive.google.com/file/d/1cbaoRMJH9y-reOesTygSvuAU7uz6dNGm/view')

assert os.path.exists(CKPT_LOCAL), 'Checkpoint not found — see instructions above.'
print('Checkpoint ready.')

Checkpoint copied from Drive: /content/FIRE/ckpt/lsunbedroom_w_adm.pt
Checkpoint ready.


## Cell 7 — Run FIRE on the Test Set


In [6]:
import sys, os
sys.path.insert(0, '/content/FIRE')

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, ImageFile
import glob

ImageFile.LOAD_TRUNCATED_IMAGES = True

# ── Simple Dataset ──────────────────────────────────────────────────────────
class SimpleImageDataset(Dataset):
    """Flat folder of images with a single label (0=real, 1=fake)."""
    def __init__(self, folder, label):
        exts = ('*.jpg', '*.jpeg', '*.png', '*.PNG', '*.JPG')
        self.paths = []
        for ext in exts:
            self.paths += glob.glob(os.path.join(folder, ext))
        self.paths = sorted(set(self.paths))
        self.label = label
        # FIRE uses ToTensor only (no normalization in its dataset.py)
        self.transform = transforms.Compose([
            transforms.Resize(256, antialias=True),
            transforms.CenterCrop(256),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.paths[idx])
            if img.mode != 'RGB':
                img = img.convert('RGB')
            return self.transform(img), self.label, os.path.basename(self.paths[idx])
        except Exception as e:
            print(f'  Warning: skipping {self.paths[idx]} — {e}')
            return self[idx + 1]

TEST_REAL = '/content/splits/test/real'
TEST_FAKE = '/content/splits/test/fake'

real_ds = SimpleImageDataset(TEST_REAL, label=0)
fake_ds = SimpleImageDataset(TEST_FAKE, label=1)

print(f'Test real images : {len(real_ds)}')
print(f'Test fake images : {len(fake_ds)}')

Test real images : 300
Test fake images : 600


In [7]:
from utils.network_utils import FIRE_model

# Load FIRE model
model = FIRE_model()

ckpt = torch.load(CKPT_LOCAL, map_location='cpu')

if isinstance(ckpt, dict) and 'state_dict' in ckpt:
    ckpt = ckpt['state_dict']
model.load_state_dict(ckpt)
model.cuda().eval()
print('FIRE model loaded and ready.')

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

... (12 outputs left out)

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://gi

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 199MB/s]


FIRE model loaded and ready.


In [8]:
from tqdm import tqdm

def run_inference(dataset, batch_size=8):
    """Run FIRE on a dataset, return (filenames, probs, true_labels)."""
    loader = DataLoader(dataset, batch_size=batch_size, num_workers=4, pin_memory=True)
    all_probs, all_labels, all_fnames = [], [], []

    with torch.no_grad():
        for imgs, labels, fnames in tqdm(loader, desc='Running FIRE'):
            imgs = imgs.cuda()
            # FIRE_model returns a tuple; first element is the logit
            out = model(imgs)[0]
            probs = out.sigmoid().squeeze(-1)  # probability of being fake
            all_probs.extend(probs.cpu().tolist())
            all_labels.extend(labels.tolist())
            all_fnames.extend(fnames)

    return all_fnames, all_probs, all_labels

from torch.utils.data import ConcatDataset
full_test_ds = ConcatDataset([real_ds, fake_ds])

print('Running FIRE inference on test set...')
fnames, y_prob, y_true = run_inference(full_test_ds, batch_size=32)

print(f'\nDone — {len(y_prob)} predictions total')
print(f'  Real: {y_true.count(0)}   Fake: {y_true.count(1)}')

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Running FIRE inference on test set...


Running FIRE:   0%|          | 0/29 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/content/FIRE/utils/network_utils.py:234: UserWarning: Casting complex values to real discards the imaginary part (Triggered internally at /pytorch/aten/src/ATen/native/Copy.cpp:308.)
  return middle_freq_image, middle_filtered_image, mask_mid_frq.to(torch.float32), mask_mid_filterd.to(torch.float32)
Running FIRE: 100%|██████████| 29/29 [02:45<00:00,  5.72s/it]


Done — 900 predictions total
  Real: 300   Fake: 600


## Cell 8 — Score Distribution Check

In [9]:
import numpy as np

real_scores = [p for p, t in zip(y_prob, y_true) if t == 0]
fake_scores = [p for p, t in zip(y_prob, y_true) if t == 1]

print(f'Real images → mean: {np.mean(real_scores):.4f}  min: {np.min(real_scores):.4f}  max: {np.max(real_scores):.4f}')
print(f'Fake images → mean: {np.mean(fake_scores):.4f}  min: {np.min(fake_scores):.4f}  max: {np.max(fake_scores):.4f}')

# Quick histogram-style summary
for threshold in [0.1, 0.3, 0.5, 0.7, 0.9]:
    real_above = sum(1 for s in real_scores if s > threshold)
    fake_above = sum(1 for s in fake_scores if s > threshold)
    print(f'  Threshold {threshold:.1f}: real_above={real_above}/{len(real_scores)}  fake_above={fake_above}/{len(fake_scores)}')

Real images → mean: 0.9882  min: 0.0000  max: 1.0000
Fake images → mean: 0.9900  min: 0.0000  max: 1.0000
  Threshold 0.1: real_above=297/300  fake_above=594/600
  Threshold 0.3: real_above=297/300  fake_above=594/600
  Threshold 0.5: real_above=296/300  fake_above=594/600
  Threshold 0.7: real_above=296/300  fake_above=594/600
  Threshold 0.9: real_above=296/300  fake_above=594/600


## Cell 9 — Compute Accuracy, AUC, F1

In [10]:
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    confusion_matrix, classification_report
)

y_pred = [1 if p >= 0.5 else 0 for p in y_prob]

acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)
f1  = f1_score(y_true, y_pred)
cm  = confusion_matrix(y_true, y_pred)

print('=' * 45)
print('  FIRE Zero-Shot Results (OpenFake test set)')
print('=' * 45)
print(f'  Accuracy : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  AUC      : {auc:.4f}')
print(f'  F1 Score : {f1:.4f}')
print('=' * 45)
print(f'\nConfusion Matrix (rows=true, cols=pred):')
print(f'               Pred Real  Pred Fake')
print(f'  True Real  :   {cm[0][0]:>5}      {cm[0][1]:>5}')
print(f'  True Fake  :   {cm[1][0]:>5}      {cm[1][1]:>5}')
print(f'\nFull Report:')
print(classification_report(y_true, y_pred, target_names=['real', 'fake']))

  FIRE Zero-Shot Results (OpenFake test set)
  Accuracy : 0.6644  (66.44%)
  AUC      : 0.5008
  F1 Score : 0.7973

Confusion Matrix (rows=true, cols=pred):
               Pred Real  Pred Fake
  True Real  :       4        296
  True Fake  :       6        594

Full Report:
              precision    recall  f1-score   support

        real       0.40      0.01      0.03       300
        fake       0.67      0.99      0.80       600

    accuracy                           0.66       900
   macro avg       0.53      0.50      0.41       900
weighted avg       0.58      0.66      0.54       900



## Cell 10 — Per-Generator Breakdown (SD vs Midjourney)


In [11]:
import pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

# Build results dataframe
results_df = pd.DataFrame({
    'filename': fnames,
    'true':     y_true,
    'prob_fake': y_prob,
    'pred':     y_pred
})

meta = pd.read_csv('/content/dataset/metadata.csv')
meta['filename'] = meta['image_path'].apply(lambda x: os.path.basename(x))
results_df = results_df.merge(meta[['filename', 'label', 'model']], on='filename', how='left')

print('=== Per-Generator Performance ===')
for gen_label in ['stable_diffusion', 'midjourney']:
    subset = results_df[results_df['label'] == gen_label]
    if len(subset) == 0:
        continue

    real_subset = results_df[results_df['label'] == 'real']
    combined = pd.concat([real_subset, subset])
    acc_g = accuracy_score(combined['true'], combined['pred'])
    try:
        auc_g = roc_auc_score(combined['true'], combined['prob_fake'])
    except:
        auc_g = float('nan')
    f1_g = f1_score(combined['true'], combined['pred'])
    print(f'\n  Generator : {gen_label}  (n={len(subset)} fake + {len(real_subset)} real)')
    print(f'  Accuracy  : {acc_g:.4f}')
    print(f'  AUC       : {auc_g:.4f}')
    print(f'  F1        : {f1_g:.4f}')


print('\n=== SD Sub-Model Breakdown (fake recall only) ===')
sd_subset = results_df[results_df['label'] == 'stable_diffusion']
for model_name, grp in sd_subset.groupby('model'):
    recall = grp['pred'].mean()
    print(f'  {model_name:30s}  n={len(grp):4d}  fake_recall={recall:.3f}')

=== Per-Generator Performance ===

  Generator : stable_diffusion  (n=296 fake + 300 real)
  Accuracy  : 0.4933
  AUC       : 0.4881
  F1        : 0.6576

  Generator : midjourney  (n=304 fake + 300 real)
  Accuracy  : 0.5099
  AUC       : 0.5133
  F1        : 0.6726

=== SD Sub-Model Breakdown (fake recall only) ===
  sd-1.5                          n=  14  fake_recall=1.000
  sd-1.5-dreamshaper              n=  21  fake_recall=1.000
  sd-1.5-epicdream                n=   9  fake_recall=1.000
  sd-2.1                          n=  67  fake_recall=0.910
  sd-3.5                          n=  68  fake_recall=1.000
  sdxl                            n=  25  fake_recall=1.000
  sdxl-epic-realism               n=  34  fake_recall=1.000
  sdxl-juggernaut                 n=  16  fake_recall=1.000
  sdxl-realvis-v5                 n=  21  fake_recall=1.000
  sdxl-touchofrealism             n=  21  fake_recall=1.000
